In [ ]:
# %%
import time
import os
import pandas as pd
import random
import json
from pathlib import Path
from tqdm.notebook import tqdm
from tqdm_joblib import tqdm_joblib

from FIT_python.pipeline_individual_id.geometric_pairwise_projection import (
    run_all_pairwise_projections_parallel,
    generate_pairwise_comparisons_from_df
)

# %%
# --- Pfade & Daten einlesen ---
data_path = Path("C:/Users/Frede/OneDrive/Projects and Disschaptors/FIT_package/data/splits/eurasian_otter_(1)")
df = pd.read_csv(data_path / "train.csv")

# %%
# --- Feature-Spalten auswählen ---
feature_cols = [
    col for col in df.columns
    if col.startswith(("dist","ang","t")) and pd.api.types.is_numeric_dtype(df[col])
]
print(f"{len(feature_cols)} numerische Features gefunden.")

# %%
# --- Parameter definieren ---
feature_counts    = list(range(2, 20))     # k = 2…19
reducers          = ["pca","lda","umap","tsne","mds","isomap"]
selection_methods = ["forward","random_forest"]
n_components_list = list(range(2, 6))       # Komponenten = 2…5
n_jobs            = 1 # Parallelisierung, 1 für Debugging -1 max speed


output_dir = Path("C:/Users/Frede/OneDrive/Projects and Disschaptors/FIT_package/results/data")
output_dir.mkdir(parents=True, exist_ok=True)

# %%
# --- Vergleiche einmal generieren & downsamplen auf 10 Pairs ---
comparisons = generate_pairwise_comparisons_from_df(
    df,
    group_sizes=[7,8],
    n_repeats=1,
    mode="both",
    selfmatch_factor=3
)
comparisons = random.sample(comparisons, 10)
print("Erste 3 Paare:", comparisons[:3])

# %%
all_results = []
timing_log  = []

# %%
# --- Sweep über selection_method mit Fortschrittsanzeigen ---
for sel in tqdm(selection_methods, desc="Selection methods"):
    t0 = time.time()
    # innere Bar für die Pairs
    with tqdm_joblib(tqdm(desc=f"Pairs ({sel})", total=len(comparisons))):
        results = run_all_pairwise_projections_parallel(
            comparisons=comparisons,
            df=df,
            feature_cols=feature_cols,
            k_features=feature_counts,       # komplette Liste k
            reducers=reducers,               # alle Reducer
            selection_method=sel,
            n_components=n_components_list,  # alle NCs
            debug=True,
            n_jobs=n_jobs
        )
    dur = time.time() - t0
    print(f"{sel}: {len(results)} Ergebnisse in {dur:.1f}s")
    # optional: erstes Ergebnis anschauen
    print("Sample:", results[:2])

    # In DataFrame und abspeichern
    df_res = pd.DataFrame(results)
    df_res["selection_method"] = sel
    fname = f"pairwise_{sel}_{len(comparisons)}pairs.csv"
    df_res.to_csv(output_dir / fname, index=False)

    all_results.append(df_res)
    timing_log.append({"selection_method": sel, "duration_s": dur, "n_results": len(results)})

# %%
# --- Timing & Gesamt-CSV ---
pd.DataFrame(timing_log).to_csv(output_dir / "timing_log.csv", index=False)

if all_results:
    full = pd.concat(all_results, ignore_index=True)
    full.to_csv(output_dir / "all_results_combined.csv", index=False)
    display(full.head())
else:
    print("Keine Ergebnisse zum Zusammenführen.")


205 numerische Features gefunden.
Erste 3 Paare: [{'ind_a': 'cibak', 'ind_b': 'paulinchen', 'size_a': 7, 'size_b': 8, 'samples_a': [327, 330, 329, 326, 325, 328, 331], 'samples_b': [251, 240, 235, 250, 241, 244, 239, 247], 'same_individual': False, 'fold': 0}, {'ind_a': 'leni', 'ind_b': 'lucka', 'size_a': 7, 'size_b': 7, 'samples_a': [184, 188, 192, 191, 189, 187, 190], 'samples_b': [166, 165, 164, 167, 161, 163, 162], 'same_individual': False, 'fold': 4}, {'ind_a': 'rosie', 'ind_b': 'sam', 'size_a': 7, 'size_b': 8, 'samples_a': [34, 32, 43, 47, 36, 27, 29], 'samples_b': [17, 11, 9, 7, 20, 12, 14, 26], 'same_individual': False, 'fold': 1}]


Selection methods:   0%|          | 0/2 [00:00<?, ?it/s]

Pairs (forward):   0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

[ERROR] pair 0 failed: n_components cannot be larger than min(n_features, n_classes - 1).


...

[ERROR] pair 7 failed: n_components cannot be larger than min(n_features, n_classes - 1).


[ERROR] pair 8 failed: n_components cannot be larger than min(n_features, n_classes - 1).





Processing Pairs: 100%|██████████| 10/10 [00:02<00:00,  4.09it/s]

[ERROR] pair 9 failed: n_components cannot be larger than min(n_features, n_classes - 1).
random_forest: 0 Ergebnisse in 2.5s
Sample: []


,selection_method


In [2]:
# nimm das erste Pair
single_pair = [comparisons[0]]

# ein einzelner Durchlauf, debug=True, n_jobs=1
res = run_all_pairwise_projections_parallel(
    comparisons     = single_pair,
    df              = df,
    feature_cols    = feature_cols,
    k_features      = [5, 10],       # z.B. nur zwei k-Werte, damit's übersichtlich bleibt
    reducers        = ["pca"],       # erstmal nur PCA
    selection_method= "forward",     # oder "random_forest"
    n_components    = [2],           # nur 2 Komponenten
    debug           = True,
    n_jobs          = 1
)

print("Result for 1 pair:", res)


Processing Pairs: 100%|██████████| 1/1 [00:00<00:00,  2.40it/s]

Result for 1 pair: [{'ind_a': 'nuernberg_2', 'ind_b': 'zewe', 'same_individual': False, 'fold': 1, 'pipeline': 'forward_k5_pca_nc2', 'selection_method': 'forward', 'reducer': 'pca', 'k_features': 5, 'n_components': 2, 'comparison_id': 0, 'center_a_x': 0.3755686441909282, 'center_a_y': -0.3268115828866957, 'center_b_x': -0.8980825494839604, 'center_b_y': -0.37277528260140624, 'center_r_x': 0.01309103415671007, 'center_r_y': 0.015143342933959945, 'coords_a_x': [-0.5134682590436048, 0.4311981514099852, 1.0711579306257857, 0.9102783756055084, 0.024009806425357105, 0.6939494880299977, 0.011855016283468522], 'coords_a_y': [0.2675869357770022, -0.2596707706257453, -0.16635397115642991, -0.9617593989887017, -0.08312774591999883, -1.252552004056872, 0.16819587476387543], 'coords_b_x': [-0.42324475087378843, -1.0206278428729556, -0.49028632351583834, -0.811540594077667, -0.9314410699391598, -1.1041691080489344, -1.2420050741143687, -1.16134563242897], 'coords_b_y': [-0.785607952396665, 0.0110598

In [3]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    balanced_accuracy_score
)
import matplotlib.pyplot as plt

# Lade kombiniertes Ergebnis
df_model = pd.read_csv(os.path.join(output_dir, "all_results_combined.csv"))

# Leere Liste für Ergebnis-Log
model_scores = []

# Iteriere über alle k-Werte
for k in sorted(df_model["k_features"].unique()):
    df_k = df_model[df_model["k_features"] == k]

    X = df_k[[
        "center_distance_ab",
        "center_distance_rcv_a",
        "center_distance_rcv_b"
    ]]
    y = df_k["same_individual"].astype(int)

    if y.nunique() < 2:
        print(f"Skipping k={k} – nur eine Klasse vorhanden.")
        continue

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )

    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    y_prob = clf.predict_proba(X_test)[:, 1]

    roc_auc = roc_auc_score(y_test, y_prob)
    bal_acc = balanced_accuracy_score(y_test, y_pred)

    print(f"\n=== k = {k} ===")
    print(classification_report(y_test, y_pred))
    print(f"ROC-AUC: {roc_auc:.3f}")
    print(f"Balanced Accuracy: {bal_acc:.3f}")

    model_scores.append({
        "k_features": k,
        "roc_auc": roc_auc,
        "balanced_accuracy": bal_acc
    })

# Zusammenfassen und plotten
scores_df = pd.DataFrame(model_scores)

plt.figure(figsize=(10, 5))
plt.plot(scores_df["k_features"], scores_df["roc_auc"], label="ROC-AUC", marker="o")
plt.plot(scores_df["k_features"], scores_df["balanced_accuracy"], label="Balanced Accuracy", marker="s")
plt.xlabel("Anzahl ausgewählter Features (k)")
plt.ylabel("Score")
plt.title("Modellperformance vs. Anzahl Features")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


KeyError: 'k_features'